In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os

In [2]:
def main():
    # Load raw data – note: variable name must be df, not data
    df = pd.read_csv(r"D:\XGBoost - Project\data\ai4i2020.csv")

    # Drop columns that are not features (including leaky labels)
    df = df.drop(columns=[
        "UDI",
        "Product ID",
        "Type",
        "TWF",
        "HDF",
        "PWF",
        "OSF",
        "RNF"
    ])

    # Engineered feature
    df["power"] = df["Torque [Nm]"] * df["Rotational speed [rpm]"]

    # Features & target
    X = df.drop(columns=["Machine failure"])
    y = df["Machine failure"]

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Create output folders
    os.makedirs("models", exist_ok=True)
    os.makedirs("data/processed", exist_ok=True)

    # Save processed data (consistent file names)
    pd.DataFrame(X_train_scaled, columns=X.columns).to_csv("data/processed/X_train.csv", index=False)
    pd.DataFrame(X_test_scaled, columns=X.columns).to_csv("data/processed/X_test.csv", index=False)
    y_train.to_csv("data/processed/y_train.csv", index=False)
    y_test.to_csv("data/processed/y_test.csv", index=False)

    # Save scaler
    joblib.dump(scaler, "models/scaler.joblib")

    print("Preprocessing complete. Data saved.")